# Import libraries

In [ ]:
import holoviews as hv
import panel as pn
import xarray as xr
from ipyleaflet import GeoJSON, LayersControl, Map, Marker, Popup, projections
from ipywidgets import HTML

In [ ]:
# pn.extension()
pn.extension("ipywidgets", sizing_mode="stretch_width")

In [ ]:
hv.extension("bokeh")

# TEST resources

In [ ]:
Resources = {
    "a": "https://thredds.met.no/thredds/dodsC/arcticdata/arctic-passion/UiT-drifters/AWS-ITO/aws_2022.nc",
    "b": "https://thredds.met.no/thredds/dodsC/arcticdata/arctic-passion/UiT-drifters/SIMBA/simba-510_air-temperature2022.nc",
    "c": "https://thredds.niva.no/thredds/dodsC/datasets/norsoop/color_fantasy/merged_acdd_color_fantasy.nc",
    "d": "https://thredds.niva.no/thredds/dodsC/datasets/nrt/color_fantasy.nc",
}

# Checks need to be made in the function so that it is confirmed that the featureType is indeed a trajectory

## ACCESS Resource ATTRIBUTES

### Check for featureType 

In [ ]:
for i in Resources:
    dsa = xr.open_dataset(Resources[i])
    if "featureType" in dsa.attrs.keys():
        print(dsa.attrs["featureType"])

In [ ]:
# example dataset (a small one)
dsa = xr.open_dataset(Resources["a"])

* Check that Time and Location can be co-located

In [ ]:
dsa.time.values.shape == dsa.latitude.values.shape

In [ ]:
dsa.dims

In [ ]:
dsa.coords

* check that `latitude` and `longitude` are both listed as `coords`

In [ ]:
set(["latitude", "longitude"]).issubset(dsa.coords)

## Plottable DATA 

need to filter out data that are not  plottable, and figure out the coordinates to build the trajectory

In [ ]:
list(dsa.variables)

In [ ]:
# remove vars without dims
plottable_vars = [i for i in dsa.data_vars.keys() if len(dsa[i].dims) >= 1]

In [ ]:
# double check matching between plottable_vars and main dataset dimension
plottable_vars = [i for i in plottable_vars if dsa[i].dims[0] in dsa.dims]

In [ ]:
var_select = pn.widgets.Select(
    name="Data Variable", options=plottable_vars, value=plottable_vars[0]
)

In [ ]:
def make_plot(variable):
    # draft tooltip (needs to check the dimension type)
    # eventually convert it to a proper datetime in case of Time Series
    TOOLTIPS = f"""
        <div>
            INDEX
            <div>
                <span style="font-size: 15px; color: #966;">[$index]</span>
            </div>

            <div>
                <span style="font-size: 15px;">Time:{variable}</span>
                <span style="font-size: 10px; color: #696;">($x, $y)</span>
            </div>
        </div>
    """
    da = dsa[variable]
    plot = hv.Curve(dsa[variable]).opts(hover_tooltips=TOOLTIPS, fontsize=20)
    return plot

In [ ]:
plot = make_plot('temperature')
interactive_plot = pn.bind(make_plot, variable=var_select)
profile_plot = pn.Column(var_select, interactive_plot)

In [ ]:
profile_plot

Now you have all the elements to create a plot for each variables. You can use the data attributes to create main title and description - or to specify axis labels and eventually units for the plotting of each vartiable.

In [ ]:
def build_line_from_xarray(ds: xr.Dataset, var_name: str) -> GeoJSON:
    """build a GeoJSON line from an xarray dataset variable"""
    lons = ds["longitude"].values
    lats = ds["latitude"].values
    coords = [[float(lon), float(lat)] for lon, lat in zip(lons, lats)]

    geojson_dict = {
        "type": "Feature",
        "properties": {"variable": var_name},
        "geometry": {"type": "LineString", "coordinates": coords},
    }

    return GeoJSON(data=geojson_dict, name=f"Line for {var_name}")

In [ ]:
build_geojson = build_line_from_xarray(dsa, "trajectory")

In [ ]:
m = Map(center=(dsa["latitude"].values[0], dsa["longitude"].values[0]), zoom=4)
m.add_layer(build_geojson)
m.add_control(LayersControl())
dsa.close()
pn.Column(profile_plot, m).servable()